# Typhoon 3.5B Distilled - Application Demo
This notebook provides a lightweight web interface to interact with the distilled 3.5B model. It uses **4-bit quantization** (`bitsandbytes`) to allow inference on free GPUs like Google Colab's T4 (16GB), requiring only ~2.5GB of VRAM.

In [ ]:
!pip install -q transformers torch accelerate bitsandbytes gradio huggingface_hub

In [ ]:
import os
from huggingface_hub import login

# Please replace with your Hugging Face Token
HF_TOKEN = "YOUR_HUGGING_FACE_TOKEN_HERE" 
login(token=HF_TOKEN)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import gradio as gr

# 1. Load Tokenizer and Model in 4-bit config
repo_id = "Phonsiri/typhoon-3.5b-cpt-ckpt"

print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(repo_id)

print("Loading Model in 4-bit (this might take a few minutes)...")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    repo_id,
    device_map="auto",
    quantization_config=quantization_config,
)
print("Model loaded successfully!")

In [ ]:
# 2. Define the Generation Function
def generate_response(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Remove the prompt from the response
    if response.startswith(prompt):
        response = response[len(prompt):].strip()
        
    return response

# 3. Launch Gradio Interface
demo = gr.Interface(
    fn=generate_response,
    inputs=gr.Textbox(lines=5, placeholder="พิมพ์ข้อความภาษาไทยที่นี่...", label="Prompt"),
    outputs=gr.Textbox(label="Generated Text"),
    title="Typhoon 3.5B Distilled (Sub-4 SLM)",
    description="โมเดลภาษาไทย 3.5B ที่ถูกสร้างขึ้นจากการกลั่นกรองความรู้ (Knowledge Distillation) จาก Typhoon 7B",
)

demo.launch(share=True, debug=True)